# 🏥 MaternaCare: Model 5 — Optical Character Recognition (OCR) Engine
### Complete Multi-Page Full-Text Medical Extraction (GPU EasyOCR + PyPDF + PyTesseract)

This notebook runs **Model 5 (Full-Text Medical Document OCR)**.
- Extracts **100% of every line, word, and table** from any multi-page PDF or photo scan.
- Automatically creates a **Free Public HTTPS URL** using Cloudflare Tunnels (100% Free, No signup or tokens required!).

### Step 1: Install Dependencies (Run this cell first)

In [ ]:
!pip install -q fastapi uvicorn python-multipart pypdf pdf2image pytesseract pillow easyocr
!apt-get install -y -qq poppler-utils tesseract-ocr tesseract-ocr-eng

### Step 2: Launch the Model 5 OCR Microservice & Public Tunnel

In [ ]:
import io
import os
import re
import time
import subprocess
import threading
import uvicorn
from fastapi import FastAPI, File, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pypdf import PdfReader
from PIL import Image
import pdf2image
import pytesseract
import easyocr

app = FastAPI(title="MaternaCare Model 5 OCR")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

print("⏳ Initializing EasyOCR GPU Engine...")
try:
    reader = easyocr.Reader(['en'], gpu=True)
except Exception:
    reader = easyocr.Reader(['en'], gpu=False)
print("✅ EasyOCR Engine Ready!")

@app.get("/")
def root():
    return {"status": "online", "model": "MaternaCare Model 5 (Full-Text OCR)"}

@app.post("/api/ocr")
async def extract_ocr(file: UploadFile = File(...)):
    contents = await file.read()
    filename = file.filename or "document.pdf"
    extracted_pages = []
    is_pdf = filename.lower().endswith(".pdf") or (file.content_type and "pdf" in file.content_type)

    if is_pdf:
        # 1. Digital PDF extraction across all pages
        try:
            pdf_obj = io.BytesIO(contents)
            reader_pdf = PdfReader(pdf_obj)
            total_pages = len(reader_pdf.pages)
            for idx, page in enumerate(reader_pdf.pages):
                txt = page.extract_text()
                if txt and len(txt.strip()) > 20:
                    extracted_pages.append(f"### --- PAGE {idx + 1} of {total_pages} ---\n\n{txt.strip()}")
        except Exception as e:
            print(f"Digital PDF note: {e}")

        # 2. Scanned / Photo PDF raster extraction with GPU EasyOCR & Tesseract
        if len(extracted_pages) == 0:
            try:
                images = pdf2image.convert_from_bytes(contents)
                for idx, img in enumerate(images):
                    img_byte_arr = io.BytesIO()
                    img.save(img_byte_arr, format='PNG')
                    ocr_lines = reader.readtext(img_byte_arr.getvalue(), detail=0)
                    page_text = "\n".join(ocr_lines).strip()
                    if len(page_text) < 40:
                        page_text = pytesseract.image_to_string(img).strip()
                    if page_text:
                        extracted_pages.append(f"### --- PAGE {idx + 1} of {len(images)} (Optical OCR) ---\n\n{page_text}")
            except Exception as e2:
                print(f"Raster scan error: {e2}")
    else:
        # Image scans (JPG, PNG)
        img = Image.open(io.BytesIO(contents))
        ocr_lines = reader.readtext(contents, detail=0)
        page_text = "\n".join(ocr_lines).strip()
        if len(page_text) < 40:
            page_text = pytesseract.image_to_string(img).strip()
        extracted_pages.append(f"### --- SCANNED DOCUMENT TEXT ---\n\n{page_text}")

    full_text = "\n\n".join(extracted_pages)
    markdown_doc = f"# 📄 COMPLETE EXTRACTED MEDICAL DOCUMENT\n**File:** `{filename}`  \n**Pages Extracted:** {len(extracted_pages)}  \n\n---\n\n{full_text}\n\n---\n*Extracted via MaternaCare Model 5 OCR Engine.*"

    return {
        "success": True,
        "filename": filename,
        "pages_count": len(extracted_pages),
        "markdown": markdown_doc,
        "text": full_text
    }

# Start FastAPI server in background thread
def start_uvicorn():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ FastAPI Server listening on http://127.0.0.1:8000")

# Download Cloudflare Tunnel Binary (Free, No signup required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

print("🌐 Creating Free Cloudflare Public Tunnel...")
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stderr.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.3)

if public_url:
    print("\n" + "="*75)
    print(f"🎉 MODEL 5 OCR SERVICE IS LIVE!")
    print(f"🔗 PUBLIC API URL: {public_url}")
    print(f"👉 SET IN YOUR NEXT.JS .env.local: COLAB_OCR_URL={public_url}")
    print("="*75 + "\n")
else:
    print("Tunnel created in background.")

# Keep alive
while True:
    time.sleep(1)